**2. Google Colab — Load the data**

In [4]:
!pip install pyspark

In [5]:
from google.colab import files

In [6]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Practice").getOrCreate()

In [9]:
uploaded = files.upload()

Saving lazy_execution_products.csv to lazy_execution_products.csv


In [10]:
sales = spark.read.csv("lazy_execution_sales_25000.csv", header=True, inferSchema=True)

In [11]:
stores = spark.read.csv("lazy_execution_stores.csv", header=True, inferSchema=True)

In [12]:
products = spark.read.csv("lazy_execution_products.csv", header=True, inferSchema=True)

**3. Exercises — Lazy Execution in PySpark**

In [13]:
from pyspark.sql.functions import col

In [14]:
#1
transaction_filtered = sales.filter(
    col('quantity') > 5
)

In [15]:
#2
selection_df = sales.select(
    'order_id',
    'product_id',
    'quantity',
    'unit_price'
)
#action
selection_df.show()

+--------+----------+--------+----------+
|order_id|product_id|quantity|unit_price|
+--------+----------+--------+----------+
| O000001|      P002|       3|      2999|
| O000002|      P014|       6|     29999|
| O000003|      P008|       6|     22999|
| O000004|      P008|       7|     22999|
| O000005|      P007|       2|     79999|
| O000006|      P005|       8|      2499|
| O000007|      P010|       8|      3999|
| O000008|      P002|       9|      2999|
| O000009|      P007|       5|     79999|
| O000010|      P006|       5|     55999|
| O000011|      P007|       1|     79999|
| O000012|      P010|       9|      3999|
| O000013|      P013|       6|      5999|
| O000014|      P005|       8|      2499|
| O000015|      P007|       7|     79999|
| O000016|      P006|       1|     55999|
| O000017|      P004|       9|     16999|
| O000018|      P012|       7|      6999|
| O000019|      P014|       5|     29999|
| O000020|      P010|       7|      3999|
+--------+----------+--------+----

In [16]:
#3
transaction_filtered = sales.filter(
    col('unit_price') > 20000
)

print("Number of matching records: ",transaction_filtered.count())

Number of matching records:  6560


In [17]:
#4
new_df = sales.withColumn(
    'gross_amount',
    col('quantity') * col('unit_price')
)

new_df.explain()

== Physical Plan ==
*(1) Project [order_id#17, order_date#18, store_id#19, product_id#20, quantity#21, unit_price#22, discount_pct#23, payment_method#24, order_status#25, sales_channel#26, (quantity#21 * unit_price#22) AS gross_amount#103]
+- FileScan csv [order_id#17,order_date#18,store_id#19,product_id#20,quantity#21,unit_price#22,discount_pct#23,payment_method#24,order_status#25,sales_channel#26] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/lazy_execution_sales_25000.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<order_id:string,order_date:date,store_id:string,product_id:string,quantity:int,unit_price:...




In [18]:
#5
transaction_filtered = sales.filter(
    col('order_status') == "Completed"
).orderBy(col('unit_price').desc())

transaction_filtered.take(10)

[Row(order_id='O000085', order_date=datetime.date(2026, 4, 22), store_id='S03', product_id='P007', quantity=4, unit_price=79999, discount_pct=0, payment_method='Card', order_status='Completed', sales_channel='Online'),
 Row(order_id='O000290', order_date=datetime.date(2026, 6, 26), store_id='S06', product_id='P007', quantity=3, unit_price=79999, discount_pct=10, payment_method='Net Banking', order_status='Completed', sales_channel='Online'),
 Row(order_id='O000098', order_date=datetime.date(2026, 4, 15), store_id='S05', product_id='P007', quantity=1, unit_price=79999, discount_pct=5, payment_method='Net Banking', order_status='Completed', sales_channel='Online'),
 Row(order_id='O000005', order_date=datetime.date(2026, 3, 14), store_id='S06', product_id='P007', quantity=2, unit_price=79999, discount_pct=0, payment_method='Cash', order_status='Completed', sales_channel='Online'),
 Row(order_id='O000107', order_date=datetime.date(2026, 5, 24), store_id='S05', product_id='P007', quantity=9

**Part B — Chained transformations**

In [19]:
#6
filters = sales.filter(
    col('order_status') == 'Completed'
)
calculation = filters.withColumn(
    'gross_amount',
    col('quantity') * col('unit_price')
)
selection = calculation.select(
    'order_id',
    'product_id',
    'quantity',
    'unit_price',
    'gross_amount'
)
sorting = selection.orderBy(col('gross_amount').desc())
#action
sorting.show()

+--------+----------+--------+----------+------------+
|order_id|product_id|quantity|unit_price|gross_amount|
+--------+----------+--------+----------+------------+
| O002603|      P007|       9|     79999|      719991|
| O006170|      P007|       9|     79999|      719991|
| O002693|      P007|       9|     79999|      719991|
| O000107|      P007|       9|     79999|      719991|
| O002841|      P007|       9|     79999|      719991|
| O000367|      P007|       9|     79999|      719991|
| O002940|      P007|       9|     79999|      719991|
| O000527|      P007|       9|     79999|      719991|
| O003108|      P007|       9|     79999|      719991|
| O000690|      P007|       9|     79999|      719991|
| O003542|      P007|       9|     79999|      719991|
| O001201|      P007|       9|     79999|      719991|
| O003884|      P007|       9|     79999|      719991|
| O001300|      P007|       9|     79999|      719991|
| O003897|      P007|       9|     79999|      719991|
| O001796|

In [20]:
#7
filtering = sales.filter(
    col('discount_pct') >= 10
)
calculation = filtering.withColumn(
    'discounted_sales_amount',
    col('quantity') * col('unit_price') - (col('quantity') * col('unit_price') * col('discount_pct') / 100)
)
displaying = calculation.sort(col('discounted_sales_amount'),ascending=False).limit(15)
#action
displaying.show()


+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+-----------------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|discounted_sales_amount|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+-----------------------+
| O006653|2026-06-29|     S01|      P007|       9|     79999|          10|          Card|   Cancelled|       Online|               647991.9|
| O006699|2026-06-25|     S05|      P007|       9|     79999|          10|   Net Banking|   Completed|       Online|               647991.9|
| O000264|2026-02-24|     S07|      P007|       9|     79999|          10|           UPI|   Completed|       Online|               647991.9|
| O007743|2026-05-22|     S08|      P007|       9|     79999|          10|   Net Banking|   Completed|     In Store|               647991.9|
| O001209|202

In [21]:
#8
from pyspark.sql.functions import sum

grouping = sales.groupBy('payment_method')
calculation = grouping.agg(
    sum(col('quantity')).alias('total_quantity')
)
sorting = calculation.orderBy(col('total_quantity').desc())
#action
sorting.show()

+--------------+--------------+
|payment_method|total_quantity|
+--------------+--------------+
|          Cash|         31754|
|          Card|         31327|
|           UPI|         30894|
|   Net Banking|         30889|
+--------------+--------------+



In [22]:
#9
from pyspark.sql.functions import count

grouping = sales.groupby('sales_channel','order_status')
calculation = grouping.agg(
    count(col('order_id')).alias('total_orders')
)
calculation.show()

+-------------+------------+------------+
|sales_channel|order_status|total_orders|
+-------------+------------+------------+
|       Online|   Completed|       10797|
|     In Store|   Cancelled|         592|
|       Online|    Returned|         987|
|       Online|   Cancelled|         633|
|     In Store|    Returned|        1013|
|     In Store|   Completed|       10978|
+-------------+------------+------------+



In [23]:
#10
new_df = sales.select(
    'order_id',
    'store_id',
    'product_id',
    'quantity'
)
new_df = new_df.drop('store_id')
new_df.first()

Row(order_id='O000001', product_id='P002', quantity=3)

**Part C — Joins and aggregations**

In [24]:
#11
joining = sales.join(
    products,
    on='product_id',
    how='inner'
)
selection = joining.select(
    'product_name',
    'category',
    'quantity',
    'unit_price'
)
selection.show(20)

+-------------------+-----------+--------+----------+
|       product_name|   category|quantity|unit_price|
+-------------------+-----------+--------+----------+
|Mechanical Keyboard|Accessories|       3|      2999|
|      Smartphone 5G|    Mobiles|       6|     29999|
|     Android Tablet|    Tablets|       6|     22999|
|     Android Tablet|    Tablets|       7|     22999|
|          Laptop 16|  Computers|       2|     79999|
|          Webcam HD|Accessories|       8|      2499|
|  Bluetooth Speaker|      Audio|       8|      3999|
|Mechanical Keyboard|Accessories|       9|      2999|
|          Laptop 16|  Computers|       5|     79999|
|          Laptop 14|  Computers|       5|     55999|
|          Laptop 16|  Computers|       1|     79999|
|  Bluetooth Speaker|      Audio|       9|      3999|
|   External HDD 2TB|    Storage|       6|      5999|
|          Webcam HD|Accessories|       8|      2499|
|          Laptop 16|  Computers|       7|     79999|
|          Laptop 14|  Compu

In [25]:
#12
joining = sales.join(
    stores,
    on='store_id',
    how='inner'
)
grouping = joining.groupby('city')
calculation = grouping.agg(
    sum(col('quantity')).alias('total_quantity')
)
sorting = calculation.orderBy(col('total_quantity').desc())
#action
sorting.show()


+---------+--------------+
|     city|total_quantity|
+---------+--------------+
|   Jaipur|         16019|
|Hyderabad|         15784|
|  Chennai|         15727|
|Bengaluru|         15718|
|  Kolkata|         15619|
|   Mumbai|         15489|
|     Pune|         15407|
|    Delhi|         15101|
+---------+--------------+



In [37]:
#13
joining = sales.join(
    products,
    on='product_id',
    how='inner'
)
all_df = joining.join(
    stores,
    on='store_id',
    how='inner'
)
filtering = all_df.filter(
    col('order_status') == 'Completed'
)
grouping = filtering.groupby('category','region')
calculation = grouping.agg(
    sum(col('quantity') * col('unit_price')).alias('total_revenue')
)

#action
calculation.show()

+-----------+------+-------------+
|   category|region|total_revenue|
+-----------+------+-------------+
|   Displays| North|     32468090|
|   Displays| South|     45761308|
|    Storage| South|     36976319|
|   Displays|  East|     13616199|
|      Audio|  East|     12043135|
|    Tablets| North|     38270336|
|    Mobiles| South|    120674524|
|    Storage|  East|     12140136|
|    Mobiles| North|     75482543|
|    Mobiles|  East|     41178126|
|Accessories|  East|      9093361|
|      Audio|  West|     25317111|
|    Tablets|  West|     37488370|
|   Displays|  West|     28286336|
|      Audio| South|     34629665|
|      Audio| North|     24327211|
|    Storage|  West|     23673341|
|  Computers| North|    240756425|
|  Computers|  West|    221092729|
|    Mobiles|  West|     80372388|
+-----------+------+-------------+
only showing top 20 rows


In [36]:
#14
filtering = sales.filter(
    col('order_status') == 'Completed'
)
joining = filtering.join(
    products,
    on='product_id',
    how='inner'
)
calculation = joining.withColumn(
    'discounted_revenue',
   col("quantity") * col("unit_price") *
    (1 - col("discount_pct") / 100)
)
grouping = calculation.groupBy(
    "product_id", "product_name"
)

total_revenue = grouping.agg(
    sum("discounted_revenue").alias("total_revenue")
)

total_revenue.orderBy(
    col("total_revenue").desc()
).show(5)

+----------+---------------+--------------------+
|product_id|   product_name|       total_revenue|
+----------+---------------+--------------------+
|      P007|      Laptop 16|5.2272546585000014E8|
|      P006|      Laptop 14| 3.830051604999999E8|
|      P014|  Smartphone 5G| 2.060406317500002E8|
|      P008| Android Tablet|1.5040656029999995E8|
|      P004|27-inch Monitor|1.1406074015000013E8|
+----------+---------------+--------------------+
only showing top 5 rows


**Part D — Understanding lazy execution**

In [46]:
#15
from pyspark.sql.functions import sum

filtering = sales.filter(
    col('order_status') == "Completed"
)
print("Data filtered")

selection = filtering.select(
    'order_id',
    'product_id',
    'quantity',
    'unit_price'
)
print("Columns selected")

add_column = selection.withColumn(
    'gross_amount',
    col('quantity') * col('unit_price')
)
print("Column created")

grouping = add_column.groupBy("product_id")
print("Grouping done")

calculation = grouping.agg(
    sum("gross_amount").alias("total_gross_amount")
)
print("Aggregation done")

sorting = calculation.orderBy(
    col("total_gross_amount").desc()
)
print("Sorted done")

# Action
sorting.show()


Data filtered
Columns selected
Column created
Grouping done
Aggregation done
Sorted done
+----------+------------------+
|product_id|total_gross_amount|
+----------+------------------+
|      P007|         552313096|
|      P006|         404480777|
|      P014|         217222759|
|      P008|         158555106|
|      P004|         120131933|
|      P015|         100484822|
|      P011|          66268636|
|      P012|          52079559|
|      P013|          44860522|
|      P010|          30048486|
|      P002|          21319891|
|      P005|          18787482|
|      P009|          14302845|
|      P003|          13591143|
|      P001|           6046033|
+----------+------------------+



In [29]:
#16
filtered = sales.filter(
    col('order_status') == 'Completed'
)
filtered.explain()


== Physical Plan ==
*(1) Filter (isnotnull(order_status#25) AND (order_status#25 = Completed))
+- FileScan csv [order_id#17,order_date#18,store_id#19,product_id#20,quantity#21,unit_price#22,discount_pct#23,payment_method#24,order_status#25,sales_channel#26] Batched: false, DataFilters: [isnotnull(order_status#25), (order_status#25 = Completed)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/lazy_execution_sales_25000.csv], PartitionFilters: [], PushedFilters: [IsNotNull(order_status), EqualTo(order_status,Completed)], ReadSchema: struct<order_id:string,order_date:date,store_id:string,product_id:string,quantity:int,unit_price:...




In [30]:
filtered.show()

+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|
| O000002|2026-05-29|     S07|      P014|       6|     29999|           0|          Card|   Completed|       Online|
| O000003|2026-06-27|     S06|      P008|       6|     22999|           0|          Card|   Completed|     In Store|
| O000004|2026-04-10|     S03|      P008|       7|     22999|          15|          Cash|   Completed|       Online|
| O000005|2026-03-14|     S06|      P007|       2|     79999|           0|          Cash|   Completed|       Online|
| O000006|2026-04-05|     S08|      P005|       8|      2499|   

In [31]:
print("Number of records: ", filtered.count())

Number of records:  21775


In [33]:
#17
joined_sales = sales.join(products, on='product_id', how='inner')
grouped = joined_sales.groupby('category').agg(
    sum(col('quantity')).alias('total_quantity')
)
grouped.collect()

[Row(category='Computers', total_quantity=16258),
 Row(category='Displays', total_quantity=7870),
 Row(category='Mobiles', total_quantity=16572),
 Row(category='Accessories', total_quantity=41839),
 Row(category='Storage', total_quantity=17251),
 Row(category='Audio', total_quantity=17159),
 Row(category='Tablets', total_quantity=7915)]

In [45]:
#18
filtering = sales.filter(
    col('order_status') == 'Completed'
)
calculation = filtering.withColumn(
    'discounted_revenue',
    col('quantity') * col('unit_price') - (col('quantity') * col('unit_price') * col('discount_pct') / 100)
)
(calculation.write.mode("overwrite")
 .parquet("/content/discounted_orders.parquet"))